# ScaleUp — Piano Transcription Server

**לפני שמתחילים:** לך ל-`Runtime → Change runtime type` ובחר **GPU (T4)**.

הרץ את 3 התאים לפי הסדר:
- **תא 1** — התקנת חבילות (~2 דק')
- **תא 2** — טעינת מודל (~3 דק')
- **תא 3** — הפעלת שרת → תקבל URL

העתק את ה-URL ל-ScaleUp → Audio to Tab → ⚙️ Advanced → MT3 Server URL

In [ ]:
#@title תא 1 — התקנת חבילות { display-mode: "form" }
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'piano-transcription-inference',
    'flask',
    'librosa',
], check=True)

print('✅ חבילות הותקנו')

In [ ]:
#@title תא 2 — טעינת המודל (~3 דקות) { display-mode: "form" }
import torch, tempfile, os, numpy as np
from piano_transcription_inference import PianoTranscription, sample_rate as PT_SR, load_audio

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print('טוען מודל... (מוריד ~130 MB בפעם הראשונה)')

TRANSCRIPTOR = PianoTranscription(device=DEVICE, checkpoint_path=None)

def transcribe_audio(audio_path: str):
    audio, _ = load_audio(audio_path, sr=PT_SR, mono=True)
    with tempfile.NamedTemporaryFile(suffix='.mid', delete=False) as tmp:
        result = TRANSCRIPTOR.transcribe(audio, tmp.name)
        os.unlink(tmp.name)
    return result['est_note_events']

def note_sequence_to_json(events) -> list:
    results = []
    for onset, offset, pitch, velocity in events:
        freq = 440.0 * (2 ** ((int(pitch) - 69) / 12.0))
        results.append({
            'startTime':  round(float(onset), 4),
            'endTime':    round(float(offset), 4),
            'midiNote':   int(pitch),
            'confidence': round(float(velocity) / 127.0, 3),
            'frequency':  round(freq, 3),
        })
    results.sort(key=lambda n: n['startTime'])
    return results

print('✅ מודל נטען!')

In [ ]:
#@title תא 3 — הפעלת שרת + URL ציבורי { display-mode: "form" }
#
# השרת רץ כל עוד הסשן פעיל.
# כל הרצה מחדש של התא יוצרת URL חדש — עדכן אותו ב-ScaleUp!

import os, tempfile, threading, subprocess, re, time, urllib.request
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.after_request
def cors(r):
    r.headers['Access-Control-Allow-Origin']  = '*'
    r.headers['Access-Control-Allow-Methods'] = 'POST,GET,OPTIONS'
    r.headers['Access-Control-Allow-Headers'] = '*'
    return r

@app.route('/health')
def health():
    return jsonify({'status': 'ok', 'model': 'piano_transcription'})

@app.route('/transcribe', methods=['POST', 'OPTIONS'])
def transcribe():
    if request.method == 'OPTIONS':
        return '', 204
    if 'audio' not in request.files:
        return jsonify({'error': 'missing audio field'}), 400
    f   = request.files['audio']
    ext = os.path.splitext(f.filename or 'audio.wav')[-1] or '.wav'
    with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as tmp:
        f.save(tmp.name)
        path = tmp.name
    try:
        print(f'[server] transcribing {f.filename} ({os.path.getsize(path)} bytes)...')
        events = transcribe_audio(path)
        notes  = note_sequence_to_json(events)
        print(f'[server] done — {len(notes)} notes')
        return jsonify({'notes': notes, 'count': len(notes)})
    except Exception as e:
        print(f'[server] ERROR: {e}')
        return jsonify({'error': str(e)}), 500
    finally:
        os.unlink(path)

# --- שחרור פורט ישן והפעלת Flask ---
os.system('fuser -k 5000/tcp 2>/dev/null || true')
time.sleep(1)
threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=5000, use_reloader=False),
    daemon=True
).start()
time.sleep(2)

# בדיקה מקומית שFlask באמת עונה
try:
    with urllib.request.urlopen('http://localhost:5000/health', timeout=5) as r:
        print('✅ Flask פעיל מקומית:', r.read().decode())
except Exception as e:
    raise RuntimeError(f'Flask לא עלה: {e}')

# --- הורדת cloudflared (פעם אחת) ---
if not os.path.exists('/content/cloudflared'):
    print('מוריד cloudflared...')
    os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared')

# --- פתיחת tunnel ---
URL_RE = re.compile(r'https://[a-z0-9-]+[.]trycloudflare[.]com')
public_url = None

proc = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://localhost:5000'],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True
)
print('פותח tunnel... (עד 30 שניות)')
deadline = time.time() + 30
for line in proc.stderr:
    m = URL_RE.search(line)
    if m:
        public_url = m.group()
        break
    if time.time() > deadline:
        break

if not public_url:
    raise RuntimeError('ה-tunnel לא עלה — הרץ את התא שוב')

# ממשיכים לקרוא את הלוג ברקע כדי שה-pipe לא ייחסם
threading.Thread(target=lambda: [None for _ in proc.stderr], daemon=True).start()

# --- בדיקה עצמית דרך ה-URL הציבורי ---
time.sleep(3)
try:
    req = urllib.request.Request(public_url + '/health', headers={'User-Agent': 'colab-selftest'})
    with urllib.request.urlopen(req, timeout=20) as r:
        body = r.read().decode()
    if '"ok"' in body:
        print('✅ בדיקה ציבורית עברה:', body)
    else:
        print('⚠️ תגובה לא צפויה מה-URL הציבורי:', body[:200])
except Exception as e:
    print(f'⚠️ ה-URL הציבורי עוד לא מגיב ({e}) — המתן 10 שניות ונסה לבדוק בדפדפן')

print()
print('=' * 55)
print('✅  השרת פעיל!')
print(f'   URL: {public_url}')
print()
print('הדבק ב-ScaleUp → Audio to Tab → ⚙️ Advanced:')
print(f'  {public_url}')
print('=' * 55)
